# EXACT 2026 — Root Data Audit: RAW vs CLEAN/V5

Notebook này audit **data gốc** và **data đang dùng v5** theo tam giác:

```text
label ↔ explanation ↔ premises/FOL/idx
```

Nguyên tắc: **không dùng Z3 như quan tòa tuyệt đối**. Notebook chỉ tìm các case mâu thuẫn/đáng review, in trực tiếp bảng chính trong output cell, và lưu CSV/JSON vào `/kaggle/working/audit_raw_vs_v5`.


In [ ]:
# ==================================================================
# STAGE 0 -- Imports & path resolver
# ==================================================================
import os, re, json, csv, math, hashlib, textwrap, statistics
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher

try:
    import pandas as pd
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=False)
    import pandas as pd

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_rows", 200)

print("="*80)
print("ROOT AUDIT NOTEBOOK -- imports OK")
print("="*80)


def _resolve(cands, label="path"):
    import glob
    expanded = []
    for p in cands:
        hits = sorted(glob.glob(p, recursive=True)) if any(ch in p for ch in "*?[") else [p]
        expanded.extend(hits)
    for p in expanded:
        if os.path.exists(p):
            print(f"  resolved {label}: {p}")
            return p
    print(f"  WARNING {label}: none found; using first candidate: {cands[0]}")
    return cands[0]

# Chồng đã cung cấp 2 path Kaggle chính xác. Các path sau chỉ là fallback.
RAW_PATH = _resolve([
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries.json",
    "/kaggle/input/**/Logic_Based_Educational_Queries.json",
    "/mnt/data/Logic_Based_Educational_Queries(2).json",
    "/mnt/data/Logic_Based_Educational_Queries.json",
], "RAW411")

CLEAN_PATH = _resolve([
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/mnt/data/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy(1).json",
    "/mnt/data/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "CLEAN/V5")

OUT_DIR = Path("/kaggle/working/audit_raw_vs_v5") if Path("/kaggle/working").exists() else Path("/mnt/data/audit_raw_vs_v5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_PATH  =", RAW_PATH)
print("CLEAN_PATH=", CLEAN_PATH)
print("OUT_DIR   =", OUT_DIR)


In [ ]:
# ==================================================================
# STAGE 1 -- Load data and flatten question-level rows
# ==================================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

raw = load_json(RAW_PATH)
clean = load_json(CLEAN_PATH)

print(f"RAW   records: {len(raw)}")
print(f"CLEAN records: {len(clean)}")

LABELS = ["A", "B", "C", "D", "YES", "NO", "UNKNOWN"]
LABEL_ORDER = {x:i for i,x in enumerate(LABELS)}


def norm_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())


def norm_text(s):
    s = norm_ws(s).lower()
    s = re.sub(r"[“”\"'`]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


def norm_answer(a):
    x = str(a).strip()
    xu = x.upper()
    if xu in ("YES", "TRUE"):
        return "YES"
    if xu in ("NO", "FALSE"):
        return "NO"
    if xu in ("UNKNOWN", "UNCERTAIN", "CANNOT BE DETERMINED", "UNDETERMINED"):
        return "UNKNOWN"
    if xu in ("A", "B", "C", "D"):
        return xu
    return xu


def h(s, n=16):
    return hashlib.md5(str(s).encode("utf-8")).hexdigest()[:n]


def record_key(rec):
    prem = "||".join(norm_text(x) for x in rec.get("premises-NL", []))
    qs = "||".join(norm_text(x) for x in rec.get("questions", []))
    return h(prem + "###" + qs, 24)


def q_key(rec, qi):
    prem = "||".join(norm_text(x) for x in rec.get("premises-NL", []))
    q = norm_text((rec.get("questions") or [""])[qi])
    return h(prem + "###" + q, 24)


def flatten(data, name):
    rows=[]
    for ri, rec in enumerate(data):
        questions = rec.get("questions", []) or []
        answers = rec.get("answers", []) or []
        exps = rec.get("explanation", []) or []
        idxs = rec.get("idx", []) or []
        for qi, q in enumerate(questions):
            ans = answers[qi] if qi < len(answers) else None
            exp = exps[qi] if qi < len(exps) else ""
            idx = idxs[qi] if qi < len(idxs) else None
            rows.append({
                "dataset": name,
                "record_i": ri,
                "q_idx": qi,
                "record_key": record_key(rec),
                "q_key": q_key(rec, qi),
                "answer": ans,
                "answer_norm": norm_answer(ans),
                "question": q,
                "explanation": exp,
                "idx": idx,
                "n_premises": len(rec.get("premises-NL", []) or []),
                "n_fol": len(rec.get("premises-FOL", []) or []),
            })
    return pd.DataFrame(rows)

raw_q = flatten(raw, "raw")
clean_q = flatten(clean, "clean")

print(f"RAW   questions: {len(raw_q)}")
print(f"CLEAN questions: {len(clean_q)}")
print("Sample raw rows:")
display(raw_q.head(3))


In [ ]:
# ==================================================================
# STAGE 2 -- Distribution comparison
# ==================================================================
def dist_table(raw_q, clean_q):
    labels = sorted(
        set(raw_q.answer_norm.dropna()) | set(clean_q.answer_norm.dropna()),
        key=lambda x: LABEL_ORDER.get(x, 999)
    )
    rows=[]
    for lab in labels:
        r = int((raw_q.answer_norm == lab).sum())
        c = int((clean_q.answer_norm == lab).sum())
        rows.append({
            "label": lab,
            "raw_count": r,
            "clean_count": c,
            "delta_clean_minus_raw": c-r,
            "raw_pct": round(100*r/max(len(raw_q),1), 2),
            "clean_pct": round(100*c/max(len(clean_q),1), 2),
        })
    return pd.DataFrame(rows)

dist = dist_table(raw_q, clean_q)
print("="*80)
print("LABEL DISTRIBUTION SHIFT")
print("="*80)
display(dist)

print("Question count delta:", len(clean_q) - len(raw_q))
print("Record count delta  :", len(clean) - len(raw))

summary = {
    "raw_records": len(raw),
    "clean_records": len(clean),
    "raw_questions": len(raw_q),
    "clean_questions": len(clean_q),
    "question_delta": len(clean_q)-len(raw_q),
    "record_delta": len(clean)-len(raw),
}

dist.to_csv(OUT_DIR / "audit_label_distribution_shift.csv", index=False)
print("Saved:", OUT_DIR / "audit_label_distribution_shift.csv")


In [ ]:
# ==================================================================
# STAGE 3 -- Match raw vs clean by premise+question key and inspect answer changes
# ==================================================================
raw_small = raw_q[["q_key","record_i","q_idx","answer_norm","answer","question","explanation","idx","record_key"]].rename(columns={
    "record_i":"raw_record_i", "q_idx":"raw_q_idx", "answer_norm":"raw_answer_norm", "answer":"raw_answer",
    "question":"raw_question", "explanation":"raw_explanation", "idx":"raw_idx", "record_key":"raw_record_key"
})
clean_small = clean_q[["q_key","record_i","q_idx","answer_norm","answer","question","explanation","idx","record_key"]].rename(columns={
    "record_i":"clean_record_i", "q_idx":"clean_q_idx", "answer_norm":"clean_answer_norm", "answer":"clean_answer",
    "question":"clean_question", "explanation":"clean_explanation", "idx":"clean_idx", "record_key":"clean_record_key"
})

matched = raw_small.merge(clean_small, on="q_key", how="outer", indicator=True)
matched["answer_changed"] = (matched["raw_answer_norm"].fillna("<MISSING>") != matched["clean_answer_norm"].fillna("<MISSING>")) & (matched["_merge"] == "both")

print("="*80)
print("RAW ↔ CLEAN QUESTION MATCHING")
print("="*80)
print("matched both       :", int((matched._merge == "both").sum()))
print("raw only removed   :", int((matched._merge == "left_only").sum()))
print("clean only added   :", int((matched._merge == "right_only").sum()))
print("answer changed     :", int(matched.answer_changed.sum()))

change_pairs = matched.loc[matched.answer_changed].groupby(["raw_answer_norm","clean_answer_norm"]).size().reset_index(name="count").sort_values("count", ascending=False)
print("Raw → Clean answer changes:")
display(change_pairs)

changes = matched.loc[matched.answer_changed, [
    "raw_record_i","raw_q_idx","clean_record_i","clean_q_idx","raw_answer_norm","clean_answer_norm",
    "raw_question","raw_explanation","clean_explanation","raw_idx","clean_idx"
]].copy()

print("Top 20 changed-answer examples:")
display(changes.head(20))

matched.to_csv(OUT_DIR / "audit_raw_clean_question_matching.csv", index=False)
changes.to_csv(OUT_DIR / "audit_label_changes_raw_vs_clean.csv", index=False)
change_pairs.to_csv(OUT_DIR / "audit_label_change_pairs.csv", index=False)
print("Saved:", OUT_DIR / "audit_label_changes_raw_vs_clean.csv")


In [ ]:
# ==================================================================
# STAGE 4 -- Explanation-vs-label heuristic flags
# These are NOT automatic fixes. They are review candidates.
# ==================================================================
def infer_mc_from_explanation(exp):
    e = norm_ws(exp)
    patterns = [
        r"support(?:ing|s)?\s+option\s+([ABCD])",
        r"option\s+([ABCD])\s+(?:is|as)\s+(?:the\s+)?(?:correct|valid|logically valid|strongest|best)",
        r"making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best)",
        r"hence,?\s+option\s+([ABCD])",
        r"therefore,?\s+option\s+([ABCD])",
        r"answer\s+is\s+([ABCD])\b",
    ]
    for pat in patterns:
        m = re.search(pat, e, flags=re.I)
        if m:
            return m.group(1).upper(), "high", f"pattern:{pat}"
    return None, None, None

POS_PHRASES = [
    "does meet all requirements", "meets all requirements", "satisfying all conditions",
    "therefore, yes", "so yes", "the statement is true", "is true", "entailed", "logically follows",
    "valid and consistent", "supporting the statement", "supports the statement"
]
NEG_PHRASES = [
    "does not meet all requirements", "doesn’t meet all requirements", "doesn't meet all requirements",
    "not meet all requirements", "so no", "therefore, no", "the statement is false", "is false",
    "cannot", "does not follow", "not enough", "isn’t confirmed", "isn't confirmed", "not confirmed",
]
UNK_PHRASES = [
    "cannot be determined", "uncertain", "unknown", "not enough information", "insufficient information",
    "not specified", "not given", "no premise confirms", "is not confirmed"
]


def infer_yn_from_explanation(exp):
    e = norm_text(exp)
    scores = {"YES":0, "NO":0, "UNKNOWN":0}
    for ph in POS_PHRASES:
        if ph in e:
            scores["YES"] += 1
    for ph in NEG_PHRASES:
        if ph in e:
            scores["NO"] += 1
    for ph in UNK_PHRASES:
        if ph in e:
            scores["UNKNOWN"] += 2
    best = max(scores, key=scores.get)
    if scores[best] == 0:
        return None, None, None
    # If unknown signal exists, prefer Unknown over generic negative wording.
    if scores["UNKNOWN"] > 0 and scores["UNKNOWN"] >= scores["NO"]:
        return "UNKNOWN", "medium", "phrase_unknown"
    return best, "low", f"phrase_scores:{scores}"


def flag_exp_label(df, name):
    rows=[]
    for _, r in df.iterrows():
        lab = r.answer_norm
        exp = r.explanation
        pred = conf = why = None
        if lab in ("A","B","C","D"):
            pred, conf, why = infer_mc_from_explanation(exp)
        elif lab in ("YES","NO","UNKNOWN"):
            pred, conf, why = infer_yn_from_explanation(exp)
        disagree = pred is not None and pred != lab
        rows.append({
            "dataset": name,
            "record_i": int(r.record_i),
            "q_idx": int(r.q_idx),
            "label": lab,
            "exp_implied": pred,
            "confidence": conf,
            "why": why,
            "flag_disagree": bool(disagree),
            "question": r.question,
            "explanation": r.explanation,
            "idx": r.idx,
        })
    return pd.DataFrame(rows)

raw_flags = flag_exp_label(raw_q, "raw")
clean_flags = flag_exp_label(clean_q, "clean")

print("="*80)
print("EXPLANATION ↔ LABEL HEURISTIC FLAGS")
print("="*80)
print("Raw flags   :", int(raw_flags.flag_disagree.sum()), "/", len(raw_flags))
print("Clean flags :", int(clean_flags.flag_disagree.sum()), "/", len(clean_flags))

print("Raw high/medium disagreement examples:")
display(raw_flags.query("flag_disagree == True and confidence in ['high','medium']").head(30))
print("Clean high/medium disagreement examples:")
display(clean_flags.query("flag_disagree == True and confidence in ['high','medium']").head(30))

raw_flags.to_csv(OUT_DIR / "audit_raw_explanation_label_flags.csv", index=False)
clean_flags.to_csv(OUT_DIR / "audit_clean_explanation_label_flags.csv", index=False)
print("Saved:", OUT_DIR / "audit_raw_explanation_label_flags.csv")
print("Saved:", OUT_DIR / "audit_clean_explanation_label_flags.csv")


In [ ]:
# ==================================================================
# STAGE 5 -- Index validity, FOL format, and record-level structural issues
# ==================================================================
def check_idx(rec, ri, dataset):
    n = len(rec.get("premises-NL", []) or [])
    issues=[]
    for qi, idx in enumerate(rec.get("idx", []) or []):
        if idx is None:
            issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_missing","idx":idx,"n_premises":n})
            continue
        if not isinstance(idx, list):
            issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_not_list","idx":idx,"n_premises":n})
            continue
        if len(idx)==0:
            issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_empty","idx":idx,"n_premises":n})
        for k in idx:
            if not isinstance(k, int):
                issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_non_int","idx":idx,"n_premises":n})
            elif k < 1 or k > n:
                issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_out_of_range","idx":idx,"bad_idx":k,"n_premises":n})
    return issues


def fol_style_stats(data, name):
    rows=[]
    for ri, rec in enumerate(data):
        fols = rec.get("premises-FOL", []) or []
        txt = " ".join(map(str, fols))
        rows.append({
            "dataset": name,
            "record_i": ri,
            "n_fol": len(fols),
            "uses_unicode_quantifier": int("∀" in txt or "∃" in txt),
            "uses_ForAll_Exists": int("ForAll" in txt or "Exists" in txt),
            "has_arithmetic": int(bool(re.search(r"[<>]=?|=\s*\d|\d", txt))),
            "has_negation": int("¬" in txt or "Not" in txt),
        })
    return pd.DataFrame(rows)

issues=[]
for i, rec in enumerate(raw):
    issues.extend(check_idx(rec, i, "raw"))
for i, rec in enumerate(clean):
    issues.extend(check_idx(rec, i, "clean"))
struct = pd.DataFrame(issues)
folstat = pd.concat([fol_style_stats(raw, "raw"), fol_style_stats(clean, "clean")], ignore_index=True)

print("="*80)
print("IDX / FOL STRUCTURAL AUDIT")
print("="*80)
print("Structural issues total:", len(struct))
if len(struct):
    display(struct.head(50))
    display(struct.groupby(["dataset","issue"]).size().reset_index(name="count"))
else:
    print("No structural idx/FOL count issues found.")

print("FOL style stats summary:")
display(folstat.groupby("dataset")[["uses_unicode_quantifier","uses_ForAll_Exists","has_arithmetic","has_negation"]].sum())

struct.to_csv(OUT_DIR / "audit_idx_fol_structural_issues.csv", index=False)
folstat.to_csv(OUT_DIR / "audit_fol_style_stats.csv", index=False)
print("Saved:", OUT_DIR / "audit_idx_fol_structural_issues.csv")
print("Saved:", OUT_DIR / "audit_fol_style_stats.csv")


In [ ]:
# ==================================================================
# STAGE 6 -- Duplicate audit
# ==================================================================
def record_signature(rec, include_answers=False):
    prem = "||".join(norm_text(x) for x in rec.get("premises-NL", []))
    qs = "||".join(norm_text(x) for x in rec.get("questions", []))
    ans = "||".join(norm_answer(x) for x in rec.get("answers", [])) if include_answers else ""
    return h(prem + "###" + qs + "###" + ans, 24)


def duplicate_groups(dataset, name):
    buckets=defaultdict(list)
    for i, rec in enumerate(dataset):
        buckets[record_signature(rec, include_answers=False)].append(i)
    rows=[]
    for sig, inds in buckets.items():
        if len(inds)>1:
            for i in inds:
                rec = dataset[i]
                rows.append({
                    "dataset": name,
                    "signature": sig,
                    "group_size": len(inds),
                    "record_i": i,
                    "answers": rec.get("answers"),
                    "q0": (rec.get("questions") or [""])[0][:200],
                    "premise0": (rec.get("premises-NL") or [""])[0][:200],
                })
    return pd.DataFrame(rows)

dup_raw = duplicate_groups(raw, "raw")
dup_clean = duplicate_groups(clean, "clean")
dups = pd.concat([dup_raw, dup_clean], ignore_index=True)

print("="*80)
print("DUPLICATE RECORD GROUPS")
print("="*80)
print("Raw duplicate rows   :", len(dup_raw))
print("Clean duplicate rows :", len(dup_clean))
if len(dups):
    display(dups.head(80))
else:
    print("No exact record-level duplicate groups found by this signature.")

dups.to_csv(OUT_DIR / "audit_duplicate_groups.csv", index=False)
print("Saved:", OUT_DIR / "audit_duplicate_groups.csv")


In [ ]:
# ==================================================================
# STAGE 7 -- High-confidence repair candidates from triangle agreement
# label ↔ explanation ↔ clean changes
# ==================================================================
# Join changed labels with raw/clean explanation flags.
raw_flag_small = raw_flags[["record_i","q_idx","exp_implied","confidence","why","flag_disagree"]].rename(columns={
    "record_i":"raw_record_i", "q_idx":"raw_q_idx", "exp_implied":"raw_exp_implied", "confidence":"raw_exp_conf", "why":"raw_exp_why", "flag_disagree":"raw_exp_disagree"
})
clean_flag_small = clean_flags[["record_i","q_idx","exp_implied","confidence","why","flag_disagree"]].rename(columns={
    "record_i":"clean_record_i", "q_idx":"clean_q_idx", "exp_implied":"clean_exp_implied", "confidence":"clean_exp_conf", "why":"clean_exp_why", "flag_disagree":"clean_exp_disagree"
})

tri = matched.merge(raw_flag_small, on=["raw_record_i","raw_q_idx"], how="left")
tri = tri.merge(clean_flag_small, on=["clean_record_i","clean_q_idx"], how="left")


def tri_reason(row):
    reasons=[]
    if row.get("answer_changed") is True:
        reasons.append("raw_clean_label_changed")
    if bool(row.get("raw_exp_disagree")):
        reasons.append("raw_label_exp_conflict")
    if bool(row.get("clean_exp_disagree")):
        reasons.append("clean_label_exp_conflict")
    if pd.notna(row.get("raw_exp_implied")) and row.get("raw_exp_implied") == row.get("clean_answer_norm") and row.get("raw_answer_norm") != row.get("clean_answer_norm"):
        reasons.append("raw_exp_supports_clean_label")
    return "; ".join(reasons)

tri["review_reason"] = tri.apply(tri_reason, axis=1)
tri_review = tri[tri["review_reason"].astype(str).str.len() > 0].copy()
tri_review = tri_review.sort_values(["answer_changed","raw_exp_disagree","clean_exp_disagree"], ascending=False)

cols = [
    "raw_record_i","raw_q_idx","clean_record_i","clean_q_idx","raw_answer_norm","clean_answer_norm",
    "raw_exp_implied","raw_exp_conf","clean_exp_implied","clean_exp_conf","review_reason",
    "raw_question","raw_explanation","clean_explanation","raw_idx","clean_idx"
]
cols = [c for c in cols if c in tri_review.columns]

print("="*80)
print("TRIANGLE REVIEW CANDIDATES")
print("="*80)
print("Review candidates:", len(tri_review))
display(tri_review[cols].head(80))

tri_review[cols].to_csv(OUT_DIR / "audit_triangle_review_candidates.csv", index=False)
print("Saved:", OUT_DIR / "audit_triangle_review_candidates.csv")


In [ ]:
# ==================================================================
# STAGE 8 -- Z3 verifier candidate list, NOT Z3 judging
# ==================================================================
# We only list cases that are structurally safer for future Z3 verification:
# - Clean dataset
# - premises-FOL exists
# - idx valid/non-empty
# - answer is Yes/No/Unknown or MC
# - avoid obvious meta questions like "fewest premises" / "strongest" where entailment alone may be insufficient.

META_PATTERNS = [
    "fewest premises", "strongest conclusion", "strongest", "best answer", "most appropriate", "current eligibility status"
]

z3_rows=[]
for _, r in clean_q.iterrows():
    qnorm = norm_text(r.question)
    is_meta = any(p in qnorm for p in META_PATTERNS)
    idx = r.idx
    idx_ok = isinstance(idx, list) and len(idx) > 0 and all(isinstance(k, int) and 1 <= k <= r.n_premises for k in idx)
    fol_ok = r.n_fol > 0
    z3_rows.append({
        "record_i": int(r.record_i),
        "q_idx": int(r.q_idx),
        "answer": r.answer_norm,
        "idx_ok": bool(idx_ok),
        "fol_ok": bool(fol_ok),
        "is_meta_question": bool(is_meta),
        "z3_verifier_priority": "high" if (idx_ok and fol_ok and not is_meta) else ("medium" if idx_ok and fol_ok else "low"),
        "question": r.question,
        "idx": r.idx,
        "explanation": r.explanation,
    })

z3cand = pd.DataFrame(z3_rows)
print("="*80)
print("Z3 VERIFIER CANDIDATE LIST -- NOT TRUTH ORACLE")
print("="*80)
display(z3cand.groupby(["z3_verifier_priority","answer"]).size().reset_index(name="count"))
print("High-priority examples:")
display(z3cand.query("z3_verifier_priority == 'high'").head(30))

z3cand.to_csv(OUT_DIR / "audit_z3_candidate_list_clean.csv", index=False)
print("Saved:", OUT_DIR / "audit_z3_candidate_list_clean.csv")


In [ ]:
# ==================================================================
# STAGE 9 -- Final summary and saved artifact list
# ==================================================================
summary.update({
    "matched_both": int((matched._merge == "both").sum()),
    "raw_only_removed": int((matched._merge == "left_only").sum()),
    "clean_only_added": int((matched._merge == "right_only").sum()),
    "answer_changed": int(matched.answer_changed.sum()),
    "raw_exp_label_flags": int(raw_flags.flag_disagree.sum()),
    "clean_exp_label_flags": int(clean_flags.flag_disagree.sum()),
    "structural_issues": int(len(struct)),
    "raw_duplicate_rows": int(len(dup_raw)),
    "clean_duplicate_rows": int(len(dup_clean)),
    "triangle_review_candidates": int(len(tri_review)),
    "z3_high_priority_candidates": int((z3cand.z3_verifier_priority == "high").sum()),
})

with open(OUT_DIR / "audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("="*80)
print("FINAL AUDIT SUMMARY")
print("="*80)
for k, v in summary.items():
    print(f"{k:32s}: {v}")

print("\nSaved artifact directory:", OUT_DIR)
for p in sorted(OUT_DIR.glob("audit_*")):
    print(" -", p.name)

print("\nNext step recommendation:")
print("1. Open audit_triangle_review_candidates.csv first.")
print("2. Manually inspect high-confidence label/explanation conflicts.")
print("3. Only after that, run Z3 verifier on audit_z3_candidate_list_clean.csv high-priority cases.")
print("4. Do NOT use Z3 alone to overwrite labels.")
